# Video2X / Real-ESRGAN — AI upscale on free Colab T4

Upscales a test clip **4x with Real-ESRGAN** (`xinntao/Real-ESRGAN`, BSD-3-Clause)
on a free Colab **T4 GPU**. Input here is a small generated clip (640×360, 2 s);
swap `test_in.mp4` for any repo clip (e.g. a 720p render) to upscale real footage.
Real-ESRGAN x4 on 720p → 2880p is heavy — for production, upscale 720p→1440p
with `scale=2` via `RealESRGAN_x2plus` instead, or tile the frames.

**Status: UNTESTED** — pins below are URL-checked (PyPI `realesrgan` 0.3.0 exists,
weights URL resolves), but no Colab run has happened yet. Report the failing cell.

Related: `Video2X` 6.4.0 ships a Linux AppImage (220 MB) — GUI-first, needs FUSE,
so the pip Real-ESRGAN path below is the Colab-friendly one. RIFE frame
interpolation (24→60 fps) is a separate optional cell at the end
(`rife-ncnn-vulkan` Ubuntu binary, 20221029).

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // (1024**2), 'MiB')

In [ ]:
print('Install Real-ESRGAN stack...')
!pip install -q --no-build-isolation realesrgan basicsr facexlib gfpgan opencv-python-headless
!python -c "import realesrgan; print('realesrgan OK')"

In [ ]:
print('Download x4plus weights (64 MB) + make a 640x360 test clip...')
!curl -sSL -o RealESRGAN_x4plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth
!ls -la RealESRGAN_x4plus.pth
!ffmpeg -y -v error -f lavfi -i testsrc=size=640x360:rate=30:duration=2 -c:v libx264 -pix_fmt yuv420p test_in.mp4
!ffprobe -v error -show_entries stream=width,height,r_frame_rate,duration -of csv test_in.mp4

In [ ]:
import cv2, torch, os
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(scale=4, model_path='RealESRGAN_x4plus.pth', model=model,
                         tile=512, tile_pad=10, pre_pad=0, half=True, device='cuda')

cap = cv2.VideoCapture('test_in.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
frames = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    out, _ = upsampler.enhance(frame, outscale=4)
    frames.append(out)
cap.release()
print('upscaled frames:', len(frames), 'shape:', frames[0].shape)

h, w = frames[0].shape[:2]
vw = cv2.VideoWriter('test_out_x4.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
for f in frames:
    vw.write(f)
vw.release()
print('saved test_out_x4.mp4')

In [ ]:
!ffprobe -v error -show_entries stream=width,height,duration -of csv test_out_x4.mp4
!ls -la test_out_x4.mp4

## Troubleshooting

- **basicsr build fails on Colab (Python 3.13):** the install cell already uses
  `--no-build-isolation` (same fix as the SadTalker notebook). If it still fails,
  pin `basicsr==1.4.2` explicitly before installing `realesrgan`.
- **CUDA out of memory:** lower `tile` to 256, or upscale at `outscale=2`.
- **Real footage:** replace `test_in.mp4` with your clip (same fps/size handling);
  720p input → 2880p output is ~4x the pixels — expect minutes per second of video.
- **RIFE interpolation (optional, next step):** download the Linux binary
  `rife-ncnn-vulkan-20221029-ubuntu.zip` from
  `github.com/nihui/rife-ncnn-vulkan/releases/tag/20221029` and run
  `./rife-ncnn-vulkan -i in.mp4 -o out60.mp4 -g 1 -f 0:2` (2x frames, GPU).
  Vulkan on Colab T4 needs the NVIDIA driver — if it errors, RIFE stays a
  local-GPU-only step and is NOT part of this notebook.